In [1]:
!pip install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 10.8 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API KEY: ")

Enter your GROQ API KEY: ··········


In [4]:
import os
from groq import Groq
from dotenv import load_dotenv

# -----------------------------------
# Load Environment Variables
# -----------------------------------
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# -----------------------------------
# Use Supported Groq Models
# -----------------------------------
ROUTER_MODEL = "llama-3.1-8b-instant"
EXPERT_MODEL = "llama-3.3-70b-versatile"

# -----------------------------------
# Define Experts (Mixture of Experts)
# -----------------------------------
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a senior technical support engineer.
Be precise, structured, and code-focused.
If debugging, clearly explain the issue and provide corrected code snippets.
Avoid emotional language."""
    },
    "billing": {
        "system_prompt": """You are a billing and subscription support specialist.
Be empathetic and professional.
Clearly explain refund policies and guide users step-by-step.
Do not provide technical debugging."""
    },
    "general": {
        "system_prompt": """You are a friendly customer support assistant.
Answer clearly and conversationally.
Keep responses helpful and easy to understand."""
    }
}

# -----------------------------------
# Router Function (Intent Classifier)
# -----------------------------------
def route_prompt(user_input: str) -> str:
    """
    Classifies user input into:
    [technical, billing, general, tool]
    Returns ONLY the category name.
    """

    router_prompt = f"""
Classify this text into one of these categories:
[technical, billing, general, tool]

Return ONLY the category name.

Text:
{user_input}
"""

    response = client.chat.completions.create(
        model=ROUTER_MODEL,
        temperature=0,  # deterministic
        messages=[
            {"role": "system", "content": "You are a strict intent classifier. Output only one word."},
            {"role": "user", "content": router_prompt}
        ],
    )

    category = response.choices[0].message.content.strip().lower()

    if category not in ["technical", "billing", "general", "tool"]:
        return "general"

    return category


# -----------------------------------
# Tool Expert (Bonus Feature)
# -----------------------------------
def get_bitcoin_price():
    """
    Mock Bitcoin price tool.
    In real implementation, call an external API.
    """
    return "The current price of Bitcoin is approximately $64,200 USD."


# -----------------------------------
# Orchestrator
# -----------------------------------
def process_request(user_input: str) -> str:

    # Step 1: Route
    category = route_prompt(user_input)
    print(f"\n[Router Decision]: {category}")

    # Step 2: Tool Handling
    if category == "tool":
        if "bitcoin" in user_input.lower():
            return get_bitcoin_price()
        else:
            return "Tool request detected, but no matching tool found."

    # Step 3: Select Expert
    system_prompt = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])["system_prompt"]

    # Step 4: Call Expert Model
    response = client.chat.completions.create(
        model=EXPERT_MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
    )

    return response.choices[0].message.content


# -----------------------------------
# Run Program
# -----------------------------------
if __name__ == "__main__":

    print("Smart Customer Support Router (MoE)")
    print("Type 'exit' to quit.\n")

    while True:
        user_query = input("Enter your query: ")

        if user_query.lower() == "exit":
            break

        result = process_request(user_query)

        print("\n[Final Response]:")
        print(result)
        print("-" * 50)

Smart Customer Support Router (MoE)
Type 'exit' to quit.

Enter your query: My Python script throws IndexError on line 5.

[Router Decision]: technical

[Final Response]:
**IndexError Debugging**

To assist with the `IndexError` on line 5 of your Python script, I'll need to know the code surrounding that line. Please provide the relevant code snippet.

That being said, here are some common causes of `IndexError`:

* Attempting to access an index that is out of range of a list or other sequence.
* Indexing a non-sequence type (e.g., trying to access an index of a number or a string that doesn't support indexing).

**Example of Error**
```python
my_list = [1, 2, 3]
print(my_list[3])  # IndexError: list index out of range
```
**Corrected Code**
```python
my_list = [1, 2, 3]
print(my_list[-1])  # Access the last element
# or
print(my_list[2])   # Access the element at index 2
```
Please provide your code, and I'll help you identify and fix the issue.
---------------------------------------